# Wyklad 6: Architektura wtyczek QGIS
### Programowanie w GIS — Kurs dla studentow I stopnia

---

> **Kurs:** Programowanie w GIS (QGIS)  
> **Wyklad:** 6 z 7  
> **Tematy:** Pakiety Pythona | Architektura wtyczki | Qt i GUI | Plugin Builder | Qt Designer  
> **Wymagania wstepne:** Wyklady 2,3,4— PyQGIS, QgsProject, QgsVectorLayer  

---

### O tym wykladzie

Ten wyklad jest czysto teoretyczny — zadnego live codingu, zadnego QGIS.
Celem jest zbudowanie solidnej **intuicji** o tym jak dziala wtyczka,
zanim zobaczymy ja w akcji na nastepnym wykladzie.

Dobra analogia jest warta wiecej niz dziesiec przykladow kodu.


---
<a id='s1'></a>

## 1. Jak Python organizuje kod — moduly i pakiety

### 1.1 Problem, ktory rozwiazujemy

Wyobraz sobie ze piszesz program majacy 10 000 linii kodu.
Trzymanie tego w jednym pliku byloby katastrofalne — nie mozna by
nic znalezc, zespol nie moze pracowac rownolegle,
a zmiana jednej rzeczy psuje dziesiec innych.

Python rozwiazuje ten problem przez **moduly** i **pakiety**.

### 1.2 Modul — jeden plik, jedna odpowiedzialnosc

**Modul** to pojedynczy plik `.py`. Kazdy plik Pythona ktory kiedykolwiek
napisales jest modulem. Nazwa pliku (bez rozszerzenia) to nazwa modulu.

Dobra praktyka: jeden modul = jedna dziedzina odpowiedzialnosci.

```
obliczenia.py       <- wszystko co liczy
wizualizacja.py     <- wszystko co rysuje
wczytywanie.py      <- wszystko co czyta pliki
zapis.py            <- wszystko co zapisuje pliki
```

Analogia: modul to jak **rozdzial w ksiazce**.
Kazdy rozdzial ma swoj temat, mozna go czytac osobno,
ale razem tworza calosc.

```python
import obliczenia
from obliczenia import funkcja_a
from obliczenia import funkcja_a, funkcja_b
```

### 1.3 Pakiet — folder modulow

Kiedy modulow robi sie duzo, grupujemy je w **pakiety**.
Pakiet to folder zawierajacy moduly i jeden obowiazkowy plik: `__init__.py`.

```
moj_program/
    __init__.py         <- bez tego to zwykly folder, nie pakiet
    obliczenia.py
    wizualizacja.py
    wczytywanie.py
    zapis.py
```

Analogia: pakiet to jak **ksiazka**.
Folder = okladka, moduly = rozdzialy, `__init__.py` = spis tresci.

Importujesz przez:

```python
import moj_projekt
from moj_projekt import obliczenia
from moj_projekt.obliczenia import funkcja_a
```

### 1.4 Pakiety zagniezdzone

Pakiety moga zawierac inne pakiety — kazdy podfolder
z `__init__.py` staje sie podpakietem.

```
moj_program/
    __init__.py
    core/                   <- podpakiet: logika biznesowa
        __init__.py
        obliczenia.py
        walidacja.py
    ui/                     <- podpakiet: interfejs uzytkownika
        __init__.py
        dialogi.py
        panele.py
    io/                     <- podpakiet: wejscie/wyjscie
        __init__.py
        wczytywanie.py
        zapis.py
```

Analogia: podpakiet to jak **czesc ksiazki** zawierajaca kilka rozdzialow.
*Czesc I: Teoria* zawiera rozdzialy 1-4,
*Czesc II: Praktyka* zawiera rozdzialy 5-8.

Importujesz przez:

```python
from moj_projekt.ui import dialogi
from moj_projekt.ui.dialogi import DialogGlowny
from moj_projekt.dane.wczytywanie import wczytaj_gpkg
```


### 1.5 Biblioteki zewnetrzne to tez pakiety

GeoPandas, NumPy, Shapely — to wszystko pakiety Pythona
zainstalowane przez `pip` w okreslonym miejscu na dysku.
Nie ma tu zadnej magii — ten sam mechanizm co Twoje pliki.

```
geopandas/
    __init__.py          <- wykonywany przy 'import geopandas'
    geodataframe.py      <- klasa GeoDataFrame
    geoseries.py         <- klasa GeoSeries
    io/
        __init__.py
        file.py          <- funkcje read_file() i to_file()
    tools/
        __init__.py
        crs.py
```

Kiedy piszesz `import geopandas as gpd`, Python:

1. Szuka folderu `geopandas/` w znanych lokalizacjach.
2. Wykonuje `geopandas/__init__.py`.
3. `__init__.py` importuje `GeoDataFrame` i udostepnia go jako `gpd.GeoDataFrame`.

Dlatego mozesz pisac `gpd.GeoDataFrame` zamiast pelnej sciezki
`gpd.geodataframe.GeoDataFrame` — `__init__.py` to sktaca.


---
<a id='s2'></a>

## 2. Rola pliku __init__.py

### 2.1 Trzy role jednego pliku

`__init__.py` pelni trzy role jednoczesnie:

**Rola 1 — Oznacza folder jako pakiet**

Sama obecnosc pliku wystarczy. Moze byc zupelnie pusty.
Bez niego Python widzi zwykly folder, nie pakiet,
i nie mozna z niego importowac.

To troche jak **szyld na drzwiach sklepu**.
Sklep moze byc pusty w srodku — szyld i tak mowi ze to sklep,
nie mieszkanie.

**Rola 2 — Kod wykonywany przy imporcie**

Kiedy ktos pisze `import moj_program`, Python wykonuje
zawartosc `__init__.py`. To swietne miejsce na:

- importy skrocone (udostepnienie kluczowych klas bezposrednio),
- inicjalizacje globalne,
- sprawdzenie wersji zaleznosci.

```python
# moj_program/__init__.py

# Wersja pakietu
WERSJA = '2.1.0'

# Import skrocony — uzytkownik pisze 'from moj_program import GlownaKlasa'
# zamiast 'from moj_program.core.silnik import GlownaKlasa'
from .core.silnik import GlownaKlasa
from .io.wczytywanie import wczytaj
from .io.zapis import zapisz
```

Analogia: `__init__.py` to jak **recepcja w biurze**.
Wchodzisz do budynku (importujesz pakiet) i recepcjonistka
kieruje Cie do odpowiednich osob (klas i funkcji),
zeby nie musial chodzic po kazdym pietrze osobno.

**Rola 3 — Kontrola publicznego API**

```python
# __init__.py
# Definiuje co jest 'publiczne' przy 'from pakiet import *'
__all__ = ['GlownaKlasa', 'wczytaj', 'zapisz']
# Reszta klas i funkcji jest 'prywatna' — do uzytku wewnetrznego
```

### 2.2 __init__.py w wtyczce QGIS

W wtyczce QGIS `__init__.py` ma jedna dodatkowa, specjalna role:
musi zawierac funkcje `classFactory()`, ktora QGIS wywoluje
przy ladowaniu wtyczki.

```python
# buffer_tool/__init__.py

def classFactory(iface):
    from .buffer_tool import BufferTool
    return BufferTool(iface)
```

To jest **umowa miedzy Twoim kodem a QGIS**:
*Daj mi obiekt `iface` (dostep do interfejsu), a ja zwroce Ci
obiekt ktory bedzie zarzadzal Twoja wtyczka.*

QGIS nie interesuje jak nazywa sie Twoja klasa, co robi
ani gdzie lezy — interesuje go tylko ze `classFactory` istnieje
i zwraca obiekt z metodami `initGui()` i `unload()`.


---
<a id='s3'></a>

## 3. Import wzgledny i bezwzgledny

### 3.1 Dwa sposoby importowania

Wewnatrz pakietu jeden modul moze importowac inny na dwa sposoby:

**Import bezwzgledny** — pelna sciezka od korzenia:

```python
from moj_program.io.wczytywanie import wczytaj
```

**Import wzgledny** — sciezka wzgledem biezacego modulu:

```python
from .wczytywanie import wczytaj     # . = biezacy folder
from ..core import GlownaKlasa       # .. = folder nadrzedny
```

### 3.2 Dlaczego wtyczki zawsze uzywaja importow wzglednych

Wyobraz sobie ze Twoja wtyczka nazywa sie `buffer_tool`.
Instalujesz ja i wszystko dziala. Potem inny uzytkownik
instaluje ja na swoim komputerze — i tez dziala.

Teraz wyobraz sobie ze uzywasz importu bezwzglednego:

```python
# buffer_tool/buffer_tool.py — ZLE
from buffer_tool.buffer_tool_dialog import BufferToolDialog
```

To dziala tylko jesli pakiet zostal zainstalowany
pod nazwa `buffer_tool`. Co jesli uzytkownik zmienil nazwe folderu?
Co jesli QGIS zaladowal go pod inna nazwa?

Import wzgledny rozwiazuje ten problem:

```python
# buffer_tool/buffer_tool.py — DOBRZE
from .buffer_tool_dialog import BufferToolDialog
```

Kropka (`.`) oznacza: *szukaj w tym samym folderze co ja*.
Nie obchodzi mnie jak ten folder sie nazywa.

Analogia: zamiast pisac na kartce
*idz do ulicy Kowalskiej 5, mieszkanie 3*,
piszesz *idz do sasiadow po lewej stronie*.
Drugi opis dziala niezaleznie od tego gdzie mieszkasz.

### 3.3 Wyjasnienie kropek

```python
from .modul import Klasa      # . = biezacy pakiet
from ..modul import Klasa     # .. = pakiet nadrzedny
from ...modul import Klasa    # ... = dwa poziomy wyzej
```

W praktyce w wtyczkach QGIS uzywa sie niemal wylacznie
pojedynczej kropki — wszystkie moduly sa w tym samym folderze.


---
<a id='s4'></a>

## 4. Czym jest wtyczka QGIS — intuicja

### 4.1 Analogia: wtyczka jak aplikacja w smartfonie

QGIS to jak system operacyjny smartfona (iOS, Android).
Dostarcza podstawowe funkcje: mapy, warstwy, narzedzia.

Wtyczka to jak **aplikacja zainstalowana ze sklepu**.
Korzysta z zasobow systemu (interfejs, dane, algorytmy),
rozszerza jego mozliwosci, ale dziala wewnatrz jego ram.

Tak samo jak aplikacja musi byc napisana zgodnie z wytycznymi
Apple lub Google, wtyczka musi spelniac wymagania QGIS.

### 4.2 Trzy sposoby rozszerzania QGIS — porownanie

| Sposob | Analogia | Interfejs | Trwalosc | Dystrybucja |
|---|---|---|---|---|
| Skrypt w konsoli | Szybka notatka na kartce | Brak | Jednorazowy | Plik .py |
| Skrypt w edytorze | Zapisany przepis | Brak | Plik .py | Plik .py |
| **Wtyczka** | **Aplikacja w sklepie** | **Wlasny GUI** | **Trwala** | **Repozytorium** |

### 4.3 Co wtyczka moze dodac do QGIS

Wtyczka moze dodac do interfejsu QGIS:

- **Przycisk w pasku narzedzi** — klikasz, cos sie dzieje.
- **Pozycje w menu** — np. `Wtyczki -> Moja Wtyczka -> Uruchom`.
- **Okno dialogowe** — formularz z polami, lista warstw, przycisk OK.
- **Panel dokowany** — staly panel po boku mapy (jak Panel warstw).
- **Algorytm w Processing Toolbox** — Twoj algorytm widoczny
  obok buforowania i innych narzedzi.
- **Nowy typ warstwy** — zaawansowane, rzadko spotykane.

### 4.4 Czego wtyczka NIE moze

Wtyczka dziala wewnatrz QGIS i jest ograniczona jego mozliwosciami.
Nie moze np.:

- zmienic fundamentalnych zasad dzialania QGIS,
- uzyskac dostepu do danych innych uzytkownikow bez ich wiedzy,
- dzialac gdy QGIS nie jest uruchomiony.

To drugie ograniczenie jest kluczowe —
dlatego do automatyzacji na serwerze uzywa sie GeoPandas i QGIS headless,
a nie wtyczek.


---
<a id='s5'></a>

## 5. Architektura aplikacji Qt

### 5.1 Czym jest Qt

QGIS jest zbudowany na bibliotece **Qt** (czytaj: *kjut*).
Qt to framework do tworzenia interfejsow graficznych,
napisany w C++, dostepny w Pythonie jako **PyQt**.

Kiedy widzisz okno QGIS — to Qt.
Panel warstw, obszar mapy, paski narzedzi, dialogi — Qt.
Twoja wtyczka rozszerza aplikacje Qt od wewnatrz.

<img src='img\Qt_designer.png' img>


### 5.2 Widgety — cegiełki interfejsu

W Qt kazdy element interfejsu to **widget**.
Widget to obiekt Pythona ktory:

- ma wizualny wyglad (rysuje sie na ekranie),
- reaguje na zdarzenia (klikniecia, wpisywanie tekstu),
- moze zawierac inne widgety.

```
QDialog (okno dialogowe)
    |
    +-- QLabel        (etykieta tekstowa: 'Warstwa wejsciowa:')
    +-- QComboBox     (lista rozwijana z warstwami)
    +-- QLabel        (etykieta: 'Promien bufora:')
    +-- QDoubleSpinBox (pole numeryczne)
    +-- QPushButton   (przycisk OK)
    +-- QPushButton   (przycisk Anuluj)
```

Analogia: widgety to jak **klocki LEGO**.
Kazdy klocek ma swoj ksztalt i kolor.
Laczysz je w dowolne kombinacje tworzac skomplikowane struktury.

### 5.3 Hierarchia widgetow

Widgety sa zorganizowane w drzewo rodzic-dziecko.
Widget rodzic:

- kontroluje pozycje i widocznosc dzieci,
- kiedy jest niszczony, niszczy rowniez swoje dzieci,
- kiedy jest ukrywany, ukrywa rowniez dzieci.

```
QMainWindow (glowne okno QGIS)
    |
    +-- QMenuBar (pasek menu)
    |       +-- QMenu ('Wtyczki')
    |               +-- QAction ('Moja Wtyczka')
    |
    +-- QToolBar (pasek narzedzi)
    |       +-- QAction (ikona wtyczki)
    |
    +-- QgsMapCanvas (obszar mapy)
    |
    +-- QDockWidget (panel)
            +-- QgsLayerTreeView (panel warstw)
```

Twoja wtyczka dokada sie do tego drzewa:
dodaje `QAction` do paska narzedzi i menu,
a po kliknieciu tworzy nowe okno dialogowe (`QDialog`).

### 5.4 Layouty — automatyczne rozmieszczanie

Gdybys recznie ustawial pozycje kazdego widgetu w pikselach,
interfejs rozpadlby sie przy zmianie rozmiaru okna.
Qt rozwiazuje to przez **layouty** — mechanizmy automatycznego rozmieszczania.

| Layout | Dzialanie |
|---|---|
| `QVBoxLayout` | Uklada widgety pionowo, jeden pod drugim |
| `QHBoxLayout` | Uklada widgety poziomo, jeden obok drugiego |
| `QGridLayout` | Uklada widgety w siatce wierszy i kolumn |
| `QFormLayout` | Uklada pary etykieta-pole (typowy formularz) |

Analogia: layout to jak **tabela w HTML** albo **siatka w CSS Grid**.
Mowisz: *te elementy maja byc w jednej kolumnie*, i Qt
sam oblicza gdzie kazdy trafi przy dowolnym rozmiarze okna.


---
<a id='s6'></a>

## 6. Cykl zycia wtyczki

### 6.1 Kiedy co sie dzieje

Wtyczka ma scisle okreslony cykl zycia. Warto go rozumiec,
bo bledy wynikajace z jego niezrozumienia sa bardzo czeste.

```
QGIS startuje
    |
    +--[1] Skanuje folder wtyczek
    |       Czyta metadata.txt kazdej wtyczki
    |       Sprawdza czy wtyczka jest wlaczona w Menedzerze
    |
    +--[2] Dla kazdej wlaczonej wtyczki:
    |       import buffer_tool            (wykonuje __init__.py)
    |       obj = classFactory(iface)     (tworzy obiekt wtyczki)
    |       obj.__init__(iface)           (konstruktor)
    |
    +--[3] obj.initGui()
    |       Wtyczka dodaje ikone do paska narzedzi
    |       Wtyczka dodaje pozycje do menu
    |
    QGIS gotowy do pracy
    |
    +--[4] Uzytkownik klika ikone wtyczki
    |       obj.run()  <- Twoja logika
    |       Pojawia sie okno dialogowe
    |
    +--[5] Uzytkownik wylacza wtyczke (lub zamyka QGIS)
            obj.unload()
            Wtyczka usuwa ikone i menu
```

### 6.2 Konstruktor __init__ — tylko jedna rola

Konstruktor `__init__` powinien robic **tylko jedno**: zachowac referencje
do `iface` i zainicjalizowac zmienne instancji.

Nie powinien:

- tworzyc okien dialogowych (okno jeszcze nie jest potrzebne),
- laczyc sie z bazami danych,
- wykonywac zadnych obliczen.

Dlaczego? Bo konstruktor jest wywolywany przy kazdym starcie QGIS,
nawet gdy uzytkownik nie zamierza uzyc wtyczki.
Ciezka operacja w konstruktorze spowalnia start QGIS.

### 6.3 initGui — budowanie interfejsu

`initGui()` jest wywolywane raz, zaraz po konstruktorze.
Tu tworzymy `QAction` (akcje = przyciski + komendy menu)
i dodajemy je do interfejsu QGIS.

```
initGui():
    stworz QAction z ikona i tekstem
    podlacz akcje do metody run()
    dodaj akcje do paska narzedzi
    dodaj akcje do menu Wtyczki
```

Wazne: w `initGui()` **nie tworzymy jeszcze dialogu**.
Dialog tworzymy dopiero w `run()`, gdy uzytkownik kliknie przycisk.
Zarzadzanie pamiecia w Qt jest wazne — niepotrzebne obiekty
powinny byc tworzone pozno i niszczone wczesnie.

### 6.4 unload — sprzatanie

`unload()` jest symetria `initGui()`.
Wszystko co zostalo dodane w `initGui()` musi byc usuniete w `unload()`.

Jesli tego nie zrobisz, przy ponownym wlaczeniu wtyczki
interfejs zapcha sie zduplikowanymi ikonami i pozycjami menu.

To jest czesty blad przy pierwszych wtyczkach:
*Dlaczego mam dwie ikony?* — bo `unload()` jest bledny lub pusty.

### 6.5 run — logika biznesowa

`run()` jest wywolywane za kazdym razem gdy uzytkownik klika przycisk.
Tu dzieje sie wszystko:

- tworzy sie lub pokazuje okno dialogowe,
- czeka sie na decyzje uzytkownika (OK lub Anuluj),
- jesli OK — pobiera sie wartosci z formularza i wykonuje operacje.

```
run():
    pokaz okno dialogowe
    czekaj na OK lub Anuluj

    jesli Anuluj:
        koniec, nie rob nic

    jesli OK:
        pobierz warstwe z listy rozwijanej
        pobierz promien z pola numerycznego
        sprawdz czy dane sa poprawne
        wykonaj buforowanie
        pokaz komunikat o sukcesie
```


---
<a id='s7'></a>

## 7. Minimalne wymagania wtyczki

### 7.1 Co QGIS sprawdza przy ladowaniu

QGIS ladujac wtyczke sprawdza kolejno:

1. Czy folder jest w katalogu wtyczek?
2. Czy istnieje `metadata.txt` z wymaganymi polami?
3. Czy istnieje `__init__.py` z funkcja `classFactory`?
4. Czy `classFactory` zwraca obiekt z metodami `initGui` i `unload`?

Brak ktoregolwiek z tych elementow = wtyczka sie nie zaladuje.

### 7.2 Minimalna struktura

```
moja_wtyczka/
    __init__.py       <- classFactory — wymagany przez QGIS
    metadata.txt      <- metadane — wymagany przez QGIS
    moja_wtyczka.py   <- klasa glowna z initGui i unload
```

Trzy pliki. Wiecej nie trzeba zeby QGIS zaladowal wtyczke.

### 7.3 Wymagane pola metadata.txt

```ini
[general]
name            = Buffer Tool
qgisMinimumVersion = 3.0
description     = Narzedzie do buforowania warstw wektorowych
version         = 0.1
author          = Jan Kowalski
email           = jan@example.com
```

Bez ktoregokolwiek z tych szesciu pol QGIS odrzuci wtyczke.

Opcjonalne ale przydatne:

```ini
about           = Dluzszy opis co wtyczka robi i dla kogo
repository      = https://github.com/uzytkownik/buffer-tool
tracker         = https://github.com/uzytkownik/buffer-tool/issues
experimental    = True
deprecated      = False
```

Pole `experimental = True` sprawia ze wtyczka jest widoczna
w Menedzerze wtyczek tylko gdy uzytkownik wlaczy opcje
*Pokazuj wtyczki eksperymentalne*. Dobrze ustawic to na `True`
podczas rozwoju, zeby przypadkowo nie trafic do repozytorium.

### 7.4 Trzy wymagane metody klasy glownej

```
class MojaWtyczka:

    __init__(self, iface)
        - zachowaj iface jako self.iface
        - zainicjalizuj zmienne (self.actions = [], self.dlg = None)
        - NIE twórz GUI tutaj

    initGui(self)
        - stworz QAction z ikona i tekstem
        - podlacz QAction do self.run
        - dodaj QAction do paska narzedzi
        - dodaj QAction do menu
        - zapamietaj QAction w self.actions

    unload(self)
        - dla kazdej akcji w self.actions:
            usun z paska narzedzi
            usun z menu
```

### 7.5 Lokalizacja folderu wtyczek

| System | Sciezka |
|---|---|
| Windows | `%APPDATA%\QGIS\QGIS3\profiles\default\python\plugins\` |
| macOS | `~/Library/Application Support/QGIS/QGIS3/profiles/default/python/plugins/` |
| Linux | `~/.local/share/QGIS/QGIS3/profiles/default/python/plugins/` |

Mozesz tez sprawdzic sciezke bezposrednio w konsoli Pythona QGIS:

```python
import qgis.utils
print(qgis.utils.plugin_paths)
```


---
<a id='s8'></a>

## 8. Plugin Builder — co generuje i dlaczego

### 8.1 Po co Plugin Builder

Pisanie minimalnej wtyczki od zera jest mozliwe ale nudne.
Plugin Builder to wtyczka do QGIS ktora generuje szkielet
nowej wtyczki — wszystkie wymagane pliki z poprawna struktura
i przykladowym kodem.

To jak **kreator nowego projektu** w IntelliJ lub VS Code:
zamiast recznie tworzyc dziesieci plikow konfiguracyjnych,
odpowiadasz na kilka pytan i dostajesz gotowy szkielet.

### 8.2 Co generuje Plugin Builder

```
buffer_tool/
    __init__.py                  <- classFactory
    buffer_tool.py               <- glowna klasa z initGui/unload/run
    buffer_tool_dialog.py        <- klasa okna dialogowego
    buffer_tool_dialog_base.ui   <- projekt GUI (do edycji w Qt Designer)
    metadata.txt                 <- metadane
    resources.py                 <- skompilowane zasoby (ikony)
    resources.qrc                <- lista zasobow (ikony, obrazki)
    icon.png                     <- domyslna ikona wtyczki
    pb_tool.cfg                  <- konfiguracja narzedzia pb_tool
    README.html                  <- wygenerowana dokumentacja
```

### 8.3 Rola kazdego pliku

**`buffer_tool.py`** — serce wtyczki.
Zawiera klase `BufferTool` z metodami `__init__`, `initGui`, `unload`, `run`.
Plugin Builder wypelnia je sensownym kodem startowym.
Tu bedziesz spedzac wiekszos czasu.

**`buffer_tool_dialog.py`** — klasa okna dialogowego.
Laduje plik `.ui` i udostepnia jego widgety jako atrybuty klasy.
Zwykle nie edytujesz tego pliku bezposrednio.

**`buffer_tool_dialog_base.ui`** — projekt interfejsu.
Plik XML opisujacy wyglad okna dialogowego.
Edytujesz go **wylacznie w Qt Designer** — nigdy recznie.

**`resources.qrc` i `resources.py`** — zasoby takie jak ikony.
Qt kompiluje obrazki do pliku `.py`, zeby byly latwo dostepne.
Jezeli zmieniasz ikone, musisz przekompilowac zasoby przez `pyrcc5`.

### 8.4 Szablon: Tool button with dialog

Plugin Builder oferuje kilka szablonow. My uzywamy
**Tool button with dialog** — najczestszy typ:

```
Ikona w pasku narzedzi
    |
    klikniecie
    |
    Okno dialogowe z formularzem
        |
        OK        -> wykonaj operacje
        Anuluj    -> zamknij bez zmian
```

Inne szablony:

| Szablon | Opis | Kiedy |
|---|---|---|
| Tool button with dialog | Przycisk + modal dialog | Wiekszosc przypadkow |
| Tool button with dock widget | Przycisk + staly panel | Narzedzia inspektujace |
| Processing provider | Nowy algorytm | Integracja z Processing |

### 8.5 Kroki w kreatorze Plugin Builder

**Ekran 1 — Podstawowe informacje:**

| Pole | Przyklad | Uwaga |
|---|---|---|
| Class name | `BufferTool` | PascalCase, bez spacji — nazwa klasy Pythona |
| Plugin name | `Buffer Tool` | Widoczna w QGIS dla uzytkownika |
| Module name | `buffer_tool` | snake_case — nazwa folderu i plikow |
| Description | `Buforowanie warstw` | Krotki opis |
| Author | Jan Kowalski | Twoje imie |
| Email | jan@example.com | Twoj email |

**Ekran 2 — Szablon:**
Wybierz *Tool button with dialog*.

**Ekran 3 — Opcje:**
Zaznacz *Flag as experimental*.
Odznacz i18n (internacjonalizacja) i testy jednostkowe.

**Kliknij Generate** — pliki pojawia sie w folderze wtyczek.


---
<a id='s9'></a>

## 9. Qt Designer — jak myslec o interfejsie

### 9.1 Czym jest Qt Designer

<img src='img\Qt_designer.png' img>

Qt Designer to graficzny edytor interfejsow Qt.
Przeciagasz elementy myszka, ustawiasz ich wlasciwosci,
i Designer generuje plik `.ui` (XML) opisujacy co narysowales.

Relacja Designer — QGIS wyglada tak:

```
Qt Designer
    |
    rysujesz okno dialogowe
    |
    zapisujesz -> buffer_tool_dialog_base.ui (XML)
    |
    Python + uic.loadUiType()
    |
    generuje klase Pythona w locie
    |
    kod Twojej wtyczki uzywa self.dlg.nazwaWidgetu
```

Dzieki temu nie musisz recznie kompilowac interfejsu —
`uic.loadUiType()` robi to automatycznie przy kazdym uruchomieniu.

### 9.2 Podstawowe widgety ktore bedziesz uzywac

| Widget | Do czego sluzy | Jak go uzyc w kodzie |
|---|---|---|
| `QLabel` | Tekst informacyjny, etykieta | `self.dlg.lblNazwa.setText('...')` |
| `QLineEdit` | Pole tekstowe (jedna linia) | `self.dlg.ledNazwa.text()` |
| `QSpinBox` | Liczba calkowita ze strzalkami | `self.dlg.spbWiek.value()` |
| `QDoubleSpinBox` | Liczba zmiennoprzec. ze strzalkami | `self.dlg.spbPromien.value()` |
| `QComboBox` | Lista rozwijana | `self.dlg.cmbTyp.currentText()` |
| `QCheckBox` | Pole wyboru tak/nie | `self.dlg.chkDolacz.isChecked()` |
| `QPushButton` | Przycisk | `self.dlg.btnWybierz.clicked.connect(...)` |
| `QProgressBar` | Pasek postepu | `self.dlg.prgPostep.setValue(50)` |
| `QgsMapLayerComboBox` | Lista warstw z projektu QGIS | `self.dlg.mMapLayerComboBox.currentLayer()` |

### 9.3 QgsMapLayerComboBox — specjalnosc QGIS

`QgsMapLayerComboBox` to widget stworzony specjalnie dla QGIS.
Automatycznie wypelnia sie warstwami z otwartego projektu
i aktualizuje gdy uzytkownik dodaje lub usuwa warstwy.

W Qt Designer nie ma go na liscie standardowych widgetow.
Dodajesz go przez mechanizm **Promote to**:

```
1. Przeciagnij zwykly QComboBox na okno
2. Kliknij prawym przyciskiem -> 'Promote to...'
3. Promoted class name: QgsMapLayerComboBox
4. Header file: qgis.gui
5. Kliknij Add -> Promote
```

Qt Designer nie wie co to jest `QgsMapLayerComboBox` —
to klasa spoza standardowej biblioteki Qt.
Mechanizm Promote mowi Designerowi:
*w pliku .ui zapisz ze to jest QgsMapLayerComboBox,
a PyQGIS przy wczytaniu sam bedzie wiedzial co to znaczy.*

### 9.4 Konwencja nazewnicza widgetow

Nazwy nadawane widgetom w Qt Designer staja sie nazwami atrybutow
w kodzie Pythona. Dobra konwencja jest wazna.

| Prefiks | Widget | Przyklad |
|---|---|---|
| `lbl` | QLabel | `lblWarstwa`, `lblPromien` |
| `led` | QLineEdit | `ledSciezka`, `ledNazwa` |
| `spb` | QSpinBox / QDoubleSpinBox | `spbPromien`, `spbLiczba` |
| `cmb` | QComboBox | `cmbTyp`, `cmbFormat` |
| `chk` | QCheckBox | `chkDolaczMale`, `chkNadpisz` |
| `btn` | QPushButton | `btnWybierzPlik`, `btnPodglad` |
| `prg` | QProgressBar | `prgPostep` |

Wtedy w kodzie piszesz:

```python
warstwa = self.dlg.mMapLayerComboBox.currentLayer()
promien  = self.dlg.spbPromien.value()
sciezka  = self.dlg.ledSciezka.text()
```

I od razu wiadomo co jest co, bez zaglądania do Qt Designer.

### 9.5 Layouty w Qt Designer

Zamiast ustawiac pozycje widgetow recznie, uzywaj layoutow.
W Qt Designer: zaznacz wszystkie widgety, potem
`Form -> Lay Out Vertically` lub `Lay Out Horizontally`.

Wtedy okno bedzie sie poprawnie zachowywac przy zmianie rozmiaru.
Bez layoutu widgety zostana na sztywno w miejscu,
a przy powiekszeniu okna pojawi sie pusta przestrzen.


---
<a id='s10'></a>

## 10. Sygnaly i sloty — komunikacja w GUI

### 10.1 Problem z GUI

Tradycyjny kod dziala sekwencyjnie — linia po linii.
GUI nie moze tak dzialac: musi **czekac na uzytkownika**.
Nie wiesz kiedy kliknie przycisk, nie wiesz w jakiej kolejnosci
wypelni formularz. Nie mozesz napisac `czekaj_az_kliknie()`.

Qt rozwiazuje to przez mechanizm **sygnalow i slotow**.

### 10.2 Sygnal i slot — intuicja

**Sygnal** to zdarzenie emitowane przez widget.
*Cos sie stalo — ktos kliknal, zmienil wartosc, wpisal tekst.*

**Slot** to funkcja wywolywana w odpowiedzi na sygnal.
*Skoro cos sie stalo, zrob to i tamto.*

Laczysz je przez `.connect()`:

```python
przycisk.clicked.connect(moja_funkcja)
# sygnal^         ^slot
```

Analogia: sygnal to jak **dzwonek u drzwi**.
Nie wiesz kiedy ktos zadzwoni — ale wiesz ze kiedy zadzwoni,
chcesz otworzyc drzwi. Podlaczycie dzwonek do otwierania drzwi
raz na poczatku — potem system sam reaguje.

### 10.3 Wiele sygnalow i slotow

Jeden sygnal moze byc podlaczony do wielu slotow.
Jeden slot moze byc podlaczony do wielu sygnalow.

```python
# Jeden przycisk -> dwie akcje
btn.clicked.connect(zapisz_wyniki)
btn.clicked.connect(odswierz_mape)

# Dwa zdarzenia -> ta sama reakcja
cmb.currentIndexChanged.connect(zaktualizuj_podglad)
spb.valueChanged.connect(zaktualizuj_podglad)
```

### 10.4 Najczesciej uzywane sygnaly w wtyczkach

| Widget | Sygnal | Kiedy jest emitowany |
|---|---|---|
| `QPushButton` | `clicked` | Po kliknieciu |
| `QCheckBox` | `stateChanged(int)` | Po zaznaczeniu/odznaczeniu |
| `QComboBox` | `currentIndexChanged(int)` | Po wyborze pozycji |
| `QDoubleSpinBox` | `valueChanged(float)` | Po zmianie wartosci |
| `QLineEdit` | `textChanged(str)` | Przy kazdym wcisnieciu klawisza |
| `QLineEdit` | `editingFinished` | Po utracie fokusu lub Enter |
| `QgsMapLayerComboBox` | `layerChanged(layer)` | Po wybraniu innej warstwy |

### 10.5 Lambda — sygnal z parametrem

Czasem chcemy przekazac dodatkowy parametr do slota.
Uzywa sie do tego funkcji lambda:

```python
# Chcemy przekazac wartosc 'gpkg' do funkcji
btn_gpkg.clicked.connect(lambda: zapisz('gpkg'))
btn_json.clicked.connect(lambda: zapisz('geojson'))
```

### 10.6 Rozlaczanie sygnalow

Sygnal mozna rozlaczyc przez `.disconnect()`.
Wazne gdy widget jest niszczony lub gdy logika sie zmienia:

```python
# Podlacz
btn.clicked.connect(moja_funkcja)

# Rozlacz konkretna funkcje
btn.clicked.disconnect(moja_funkcja)

# Rozlacz wszystko
btn.clicked.disconnect()
```

### 10.7 messageBar — wlasciwy sposob komunikacji

W skryptach uzywalismy `print()`. W wtyczkach to zly pomysl —
uzytkownik nie widzi konsoli Pythona.

QGIS ma wbudowany **pasek komunikatow** (messageBar) —
kolorowy pasek pojawiajacy sie na gorze obszaru mapy.

```
Poziom           Kolor     Kiedy uzywac
---
pushInfo         niebieski  Informacja (operacja rozpoczeta)
pushSuccess      zielony    Sukces (operacja zakonczona)
pushWarning      zolty      Ostrzezenie (cos moze byc zle)
pushCritical     czerwony   Blad (operacja sie nie powiodla)
```

Komunikat znika samoczynnie po kilku sekundach lub
uzytkownik moze go zamknac recznie.
To standard UX w QGIS — uzytkownik jest przyzwyczajony do tego paska.


---
<a id='s11'></a>

## 11. Podsumowanie — mapa pojec

### 11.1 Hierarchia tego co budujemy

```
Python
    |
    +-- Modul (.py)              pojedynczy plik z kodem
    |
    +-- Pakiet (folder + __init__.py)
            |
            +-- Wtyczka QGIS    pakiet ze specjalnymi wymaganiami
                    |
                    +-- classFactory    umowa z QGIS
                    +-- initGui         budowanie GUI
                    +-- unload          sprzatanie
                    +-- run             logika
                    +-- Qt Dialog       okno z formularzem
                            |
                            +-- Widgety (QLabel, QSpinBox, ...)
                            +-- Sygnaly i sloty
```

### 11.2 Narzedzia ktorych uzywamy

| Narzedzie | Rola | Kiedy |
|---|---|---|
| **Plugin Builder** | Generuje szkielet wtyczki | Raz, na poczatku |
| **Qt Designer** | Projektuje GUI (plik .ui) | Przy kazdej zmianie interfejsu |
| **Plugin Reloader** | Przeladowuje wtyczke bez restartu QGIS | Przy kazdej zmianie kodu |
| **Menedzer wtyczek** | Wlacza/wylacza wtyczke | Przy pierwszym uruchomieniu |

### 11.3 Slownik pojec

| Pojecie | Definicja |
|---|---|
| **Modul** | Pojedynczy plik `.py` |
| **Pakiet** | Folder z modulami i plikiem `__init__.py` |
| **`__init__.py`** | Plik inicjalizujacy pakiet; w wtyczce musi zawierac `classFactory` |
| **Import wzgledny** | Import uzywajacy kropki (`.modul`) — niezalezny od nazwy pakietu |
| **`classFactory`** | Funkcja wywolywana przez QGIS — musi zwrocic obiekt wtyczki |
| **`initGui`** | Metoda dodajaca przyciski i menu do interfejsu QGIS |
| **`unload`** | Metoda usuwajaca elementy GUI — symetria initGui |
| **Qt** | Framework do tworzenia GUI, na ktorym zbudowany jest QGIS |
| **Widget** | Element interfejsu Qt (przycisk, pole tekstowe, lista) |
| **Layout** | Mechanizm automatycznego rozmieszczania widgetow |
| **Sygnal** | Zdarzenie emitowane przez widget (np. klikniecie) |
| **Slot** | Funkcja wywolywana w odpowiedzi na sygnal |
| **`.connect()`** | Laczenie sygnalu ze slotem |
| **Plugin Builder** | Wtyczka QGIS generujaca szkielet nowej wtyczki |
| **Qt Designer** | Graficzny edytor interfejsow Qt, generuje pliki `.ui` |
| **Plik `.ui`** | XML opisujacy wyglad okna — edytowany w Qt Designer |
| **Promote to** | Mechanizm Qt Designer zastepujacy widget klasa pochodna |
| **`QgsMapLayerComboBox`** | Widget QGIS — lista rozwijana z warstwami projektu |
| **messageBar** | Pasek komunikatow QGIS — zamiast print() w wtyczkach |
| **Plugin Reloader** | Wtyczka przeladowujaca inna wtyczke bez restartu QGIS |


---
*Wyklad 6 — Programowanie w GIS (QGIS) — studia inżynierskie.*
